# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (see code below).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print high-level info
print(f"Dataset name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"RecordSet count: {len(dataset.metadata.recordSet)}")

## 2. Data Overview
Review available record sets, fields, their `@id`s and descriptions.

In [ ]:
# List all record sets with their @id, name, and description
from mlcroissant._src.structure_graph.base_node import get_node_field

record_sets = dataset.metadata.recordSet  # This is a list of record set objects
print(f"Number of record sets: {len(record_sets)}\n")

overview = []
for rec in record_sets:
    print(f"RecordSet: @id={rec['@id']}")
    print(f"  name: {rec.get('name','')}")
    print(f"  description: {rec.get('description','')}")
    if 'field' in rec:
        print("  Fields:")
        for f in rec['field']:
            # f is a field object
            fn = f.get('name', '')
            print(f"    @{f['@id']}: {fn}, type: {f.get('dataType','')}")
    print()

## 3. Data Extraction
Load data from the main record set(s) into pandas DataFrames for analysis using their `@id`. 

⚠️ Note: Replace the sample record set `@id` below (`@id` should be copied from those listed in the section above).

In [ ]:
# For demonstration, we select the main dataset record set.
# If the dataset has only one record set, use its @id. Otherwise, list all.
main_recordsets = [rec['@id'] for rec in dataset.metadata.recordSet]
print(f"Main record set @ids: {main_recordsets}")

dataframes = {}
for record_set_id in main_recordsets:
    print(f"Loading records for {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"DataFrame columns: {dataframes[record_set_id].columns.tolist()}")

# Preview the main dataframe
main_df_id = main_recordsets[0]
dataframes[main_df_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping. 

Please refer to the data overview above to pick the correct field `@id`s. Below is a generic template with an example assuming numeric field `'cr:field:mw'` (molecular weight) and grouping by `'cr:field:sample_id'`. Adjust the field IDs to match your dataset as needed.

In [ ]:
# Example: EDA on a numeric field
# Replace these IDs with real ones from your dataset

record_set_id = main_df_id  # Main record set @id
df = dataframes[record_set_id]

# Try guessing a numeric column by name ("MW" or "molecular_weight")
# Or replace with e.g. 'cr:field:mw' or 'cr:field:molecular_weight' 
numeric_field_candidates = [c for c in df.columns if 'mw' in c.lower() or 'weight' in c.lower() or 'abundance' in c.lower()] 
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
    print(f"Will use numeric field '{numeric_field}'")
else:
    numeric_field = df.columns[0]
    print(f"No weight/abundance column found, defaulting to {numeric_field}")

# Set a meaningful filter threshold (e.g. MW > 10000)
try:
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
except Exception:
    pass

threshold = df[numeric_field].quantile(0.75)  # Top 25% as an arbitrary threshold
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered to records with {numeric_field} > {threshold}")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[numeric_field + "_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, numeric_field+"_normalized"]].head())

# Group by a categorical field, e.g. 'sample' or 'accession'
group_field_candidates = [c for c in df.columns if 'sample' in c.lower() or 'accession' in c.lower() or 'group' in c.lower()]
if group_field_candidates:
    group_field = group_field_candidates[0]
    print(f"Grouping by field '{group_field}'")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or field relationships. Here, a histogram and a boxplot of the numeric field are shown.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field}")
plt.show()

# Boxplot by group if group_field is available
if group_field_candidates:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df.dropna(subset=[group_field, numeric_field]))
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
We have loaded and explored the FAIR² Mass Spectrometry dataset with `mlcroissant`. Using the Croissant schema, we illustrated how to identify record sets and fields by their `@id`, load the associated records, and perform basic filtering, normalization, grouping, and visualization. This foundation lets you build further analyses, such as finding patterns in protein expression, comparing samples, or preparing the data for machine learning tasks.

You are encouraged to use the record set and field `@id`s identified above for more advanced queries and analyses specific to your scientific questions!